In [ ]:

# ==========================================
# KNN CLASSIFICATION WITH ROC ANALYSIS(general)
# ==========================================

# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score
)

# ==========================================
# 2. Load dataset
# ==========================================

file_path = "/content/your_dataset.csv"
target_column = "target"

df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
print(df.head())

# Remove rows where target is missing
df = df.dropna(subset=[target_column])

# Separate features and target
X = df.drop(columns=[target_column])
y = df[target_column]

# Remove likely ID columns (optional)
# X = X.drop(columns=["id"], errors="ignore")

# ==========================================
# 3. Encode target labels
# ==========================================

le = LabelEncoder()
y = le.fit_transform(y)

print("\nTarget Classes:", le.classes_)

# ==========================================
# 4. Identify numerical and categorical columns
# ==========================================

numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(exclude=np.number).columns

# Numerical preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# ==========================================
# 5. Split dataset
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==========================================
# 6. Find the best K value
# ==========================================

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("knn", KNeighborsClassifier())
])

# Use only K values supported by the training set
max_k = min(15, len(X_train))

k_values = [k for k in range(1, max_k + 1, 2)]

if not k_values:
    raise ValueError("Not enough training samples for KNN.")

param_grid = {
    "knn__n_neighbors": k_values
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("\nBest K:", grid.best_params_["knn__n_neighbors"])

# ==========================================
# 7. Make predictions
# ==========================================

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)

# ==========================================
# 8. Evaluate model
# ==========================================

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", accuracy)
print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(le.classes_)),
    target_names=[str(c) for c in le.classes_],
    zero_division=0
))

# ==========================================
# 9. Confusion Matrix
# ==========================================

cm = confusion_matrix(
    y_test, y_pred,
    labels=np.arange(len(le.classes_))
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("KNN Confusion Matrix")
plt.show()

# ==========================================
# 10. ROC Curve and AUC
# ==========================================

n_classes = len(le.classes_)

if n_classes == 2:

    # Binary classification
    fpr, tpr, thresholds = roc_curve(y_test, y_prob[:, 1])

    auc_score = roc_auc_score(y_test, y_prob[:, 1])

    plt.plot(
        fpr, tpr,
        label=f"KNN (AUC = {auc_score:.3f})"
    )

else:

    # Multiclass classification
    y_test_bin = label_binarize(
        y_test,
        classes=np.arange(n_classes)
    )

    auc_score = roc_auc_score(
        y_test_bin,
        y_prob,
        average="macro",
        multi_class="ovr"
    )

    for i, class_name in enumerate(le.classes_):
        fpr, tpr, _ = roc_curve(
            y_test_bin[:, i],
            y_prob[:, i]
        )

        class_auc = roc_auc_score(
            y_test_bin[:, i],
            y_prob[:, i]
        )

        plt.plot(
            fpr, tpr,
            label=f"{class_name} (AUC = {class_auc:.3f})"
        )

    print("Macro ROC-AUC:", auc_score)

# Plot ROC
plt.plot([0, 1], [0, 1], "k--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("KNN ROC Curve")
plt.legend()
plt.grid()
plt.show()

print("\nFinal ROC-AUC Score:", auc_score)

# ==========================================
# 11. Accuracy for different K values
# ==========================================

k_accuracy = []

for k in k_values:
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

    model.fit(X_train, y_train)
    prediction = model.predict(X_test)

    k_accuracy.append(accuracy_score(y_test, prediction))

plt.plot(k_values, k_accuracy, marker="o")
plt.xlabel("Number of Neighbors (K)")
plt.ylabel("Accuracy")
plt.title("KNN Accuracy for Different K Values")
plt.grid()
plt.show()
